# 04 — Baseline CNN: Vehicle Damage Classification

**Task 1 of the pipeline.** Train a small convolutional network *from scratch* to classify a vehicle image as `Condition = 1` (damaged) or `Condition = 0` (undamaged). This is the reference point against which the pretrained ResNet50 (notebook 05) is judged.

From a modelling standpoint this network estimates a *claim-incidence probability* `P(damaged | image)` — the frequency leg of the classical frequency × severity decomposition used in claim reserving. The severity leg (claim `Amount`) is handled by the regression notebooks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
DATA_DIR = Path("c:\\Users\\Niraj Mhatre\\projects\\motor_insurance_data\\Fast_Furious_Insured")

TRAIN_IMG_DIR = DATA_DIR / "trainImages"
TRAIN_METADATA = DATA_DIR / "processed" / "train_metadata_clean.csv"

MODEL_DIR = Path("../models")
TABLE_DIR = Path("../outputs/tables")
PRED_DIR = Path("../outputs/predictions")

for d in (MODEL_DIR, TABLE_DIR, PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(TRAIN_METADATA)
print(df.shape)
df.head()

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["Condition"],
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print(train_df["Condition"].value_counts(normalize=True))

In [ ]:
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
class VehicleDamageDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(self.image_dir / row["Image_path"]).convert("RGB")
        label = torch.tensor(row["Condition"], dtype=torch.float32)
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
BATCH_SIZE = 32

train_dataset = VehicleDamageDataset(train_df, TRAIN_IMG_DIR, train_transform)
val_dataset = VehicleDamageDataset(val_df, TRAIN_IMG_DIR, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

## Architecture

Three convolutional blocks (Conv → BatchNorm → ReLU → MaxPool), followed by global average pooling and a single linear unit. A logit is returned; the sigmoid is folded into `BCEWithLogitsLoss` for numerical stability.

In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(128, 1)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.classifier(x)


model = BaselineCNN().to(device)
model

In [ ]:
num_negative = (train_df["Condition"] == 0).sum()
num_positive = (train_df["Condition"] == 1).sum()

pos_weight = torch.tensor([num_negative / num_positive], dtype=torch.float32).to(device)
print("Positive class weight:", pos_weight.item())

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_labels, all_probs = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)

            probs = torch.sigmoid(outputs)
            all_probs.extend(probs.cpu().numpy().ravel())
            all_labels.extend(labels.cpu().numpy().ravel())

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    preds = (all_probs >= 0.5).astype(int)

    return {
        "loss": running_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_labels, preds),
        "precision": precision_score(all_labels, preds, zero_division=0),
        "recall": recall_score(all_labels, preds, zero_division=0),
        "f1": f1_score(all_labels, preds, zero_division=0),
        "auc": roc_auc_score(all_labels, all_probs),
        "pr_auc": average_precision_score(all_labels, all_probs),
        "probs": all_probs,
        "labels": all_labels,
    }

In [ ]:
EPOCHS = 20
best_f1 = 0.0
history = []

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    m = evaluate(model, val_loader, criterion)
    scheduler.step(m["f1"])

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"train {train_loss:.4f} | val {m['loss']:.4f} | "
        f"F1 {m['f1']:.4f} | P {m['precision']:.4f} | "
        f"R {m['recall']:.4f} | AUC {m['auc']:.4f}"
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": m["loss"],
        "accuracy": m["accuracy"],
        "precision": m["precision"],
        "recall": m["recall"],
        "f1": m["f1"],
        "auc": m["auc"],
        "pr_auc": m["pr_auc"],
    })

    if m["f1"] > best_f1:
        best_f1 = m["f1"]
        torch.save(model.state_dict(), MODEL_DIR / "baseline_cnn.pth")
        print("  -> best baseline CNN saved")

## Learning curves

In [ ]:
hist = pd.DataFrame(history)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].plot(hist["epoch"], hist["train_loss"], marker="o", label="train")
ax[0].plot(hist["epoch"], hist["val_loss"], marker="o", label="val")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("BCE loss")
ax[0].set_title("Loss")
ax[0].legend()

ax[1].plot(hist["epoch"], hist["f1"], marker="o", label="F1")
ax[1].plot(hist["epoch"], hist["auc"], marker="o", label="ROC-AUC")
ax[1].set_xlabel("Epoch")
ax[1].set_title("Validation metrics")
ax[1].legend()

plt.tight_layout()
plt.show()

## Final evaluation on the held-out validation split

In [ ]:
model.load_state_dict(torch.load(MODEL_DIR / "baseline_cnn.pth", map_location=device))
final = evaluate(model, val_loader, criterion)

print(classification_report(final["labels"], (final["probs"] >= 0.5).astype(int), digits=4))

In [ ]:
cm = confusion_matrix(final["labels"], (final["probs"] >= 0.5).astype(int))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax.text(j, i, str(v), ha="center", va="center")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Undamaged", "Damaged"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Undamaged", "Damaged"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Baseline CNN — confusion matrix")
plt.colorbar(im)
plt.tight_layout()
plt.show()

## Threshold analysis

The default 0.5 cut-off is rarely optimal under class imbalance. In a claims-triage setting a false negative (missing a genuinely damaged vehicle) and a false positive carry different costs, so we inspect precision and recall across thresholds and record the F1-optimal operating point.

In [ ]:
prec, rec, thr = precision_recall_curve(final["labels"], final["probs"])
f1_curve = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = int(np.nanargmax(f1_curve[:-1]))
best_threshold = float(thr[best_idx])

print("F1-optimal threshold:", round(best_threshold, 4))
print("F1 at that threshold:", round(float(f1_curve[best_idx]), 4))

plt.figure(figsize=(7, 5))
plt.plot(thr, prec[:-1], label="precision")
plt.plot(thr, rec[:-1], label="recall")
plt.plot(thr, f1_curve[:-1], label="F1")
plt.axvline(best_threshold, color="k", ls="--", lw=1)
plt.xlabel("Threshold")
plt.title("Baseline CNN — precision / recall / F1 vs threshold")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
metrics_row = pd.DataFrame([{
    "model": "Baseline CNN",
    "accuracy": final["accuracy"],
    "precision": final["precision"],
    "recall": final["recall"],
    "f1": final["f1"],
    "roc_auc": final["auc"],
    "pr_auc": final["pr_auc"],
    "best_threshold": best_threshold,
}])
metrics_row.to_csv(TABLE_DIR / "baseline_cnn_val_metrics.csv", index=False)

val_out = val_df.reset_index(drop=True).copy()
val_out["cnn_baseline_prob"] = final["probs"]
val_out[["Image_path", "Condition", "cnn_baseline_prob"]].to_csv(
    PRED_DIR / "baseline_cnn_val_preds.csv", index=False
)

metrics_row

The from-scratch CNN gives an honest lower bound. Notebook 05 asks whether an ImageNet-pretrained ResNet50, adapting representations it already holds, buys a materially better claim-incidence signal on the same split and metrics.